In [0]:
import os
import requests
from datetime import datetime

date = datetime.now().strftime("%Y%m%d")

SOURCE_URL = "https://www.ispdados.rj.gov.br/Arquivos/BaseEstadoTaxaMes.csv"
CATALOG_NAME = "isp"
SCHEMA_NAME = "bronze"
VOLUME_NAME = "isp_volumes"
FOLDER_NAME = "raw"
FILE_NAME = f"timeseries_monthly_since_2003_{date}.csv"

volume_dir = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/{FOLDER_NAME}"
target_file_path = os.path.join(volume_dir, FILE_NAME)

In [0]:
os.makedirs(volume_dir, exist_ok=True)

def download_file(url: str, dest_path: str, chunk_size: int = 8192) -> None:
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(dest_path, "wb") as output_file:
            for chunk in response.iter_content(chunk_size):
                if chunk:
                    output_file.write(chunk)
        
        print(f"File downloaded successfully to: {dest_path}")
    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"Failed to download file from {url}: {error}") from error

download_file(SOURCE_URL, target_file_path)

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .option("delimiter", ";")
        .load(target_file_path)
)

display(df)